In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from sklearn.preprocessing import LabelEncoder


In [2]:
synthetic_df = pd.read_csv('/Users/pin.lyu/Documents/BC_Folder/NLP/Data/synthetic_text.csv')

synthetic_df

,Unnamed: 0,EMAIL,USERNAME,ID_NUM,PHONE_NUM,URL_PERSONAL,STREET_ADDRESS,ESSAY
0,0,['lisa08@gmail.com'],['carrie26'],"['711801320', '522 AYO']",['+1-875-586-8809x1891'],['https://twitter.com/amanda96'],"['9930 Joy Hollow Suite 517\nSherriport, WI 53...",The digital age has blurred the lines between ...
1,1,['tammy76@yahoo.com'],"['nhoward', 'juancampos']",['Z89-24I'],['515-978-1565'],"['https://twitter.com/qgrimes', 'https://twitt...","['097 Sanchez Islands Apt. 393\nPort Tammy, AS...",The digital age has irrevocably woven itself i...
2,2,"['bosborne@gmail.com', 'crystalgarcia@hotmail....","['james71', 'bishoptanner', 'debra94']",['418 3MZ'],"['2679537510', '608-399-3318x868']",['https://instagram.com/ronaldknight'],"['PSC 1611, Box 6207\nAPO AA 90471', '4151 Mic...",The question of identity in the digital age is...
3,3,['yfigueroa@yahoo.com'],['scott92'],['161-16-1975'],['(769)972-8457x6377'],['https://twitter.com/wallacedouglas'],[],The flickering fluorescent lights of the unive...
4,4,"['donnadennis@gmail.com', 'carterhannah@hotmai...",['michelelopez'],['DPLW42035574485291'],['789-542-6223'],['https://facebook.com/karicarter'],"['90611 Robert Plaza\nYangberg, OR 55838']",## The Unexpected Detour: Finding Community in...
...,...,...,...,...,...,...,...,...
1995,1995,[],['pamelagibson'],['KKVJ56034532166723'],['237-590-1908x664'],['https://facebook.com/brian33'],[],The flickering screen illuminated my face as I...
1996,1996,[],['msmith'],['I02374580'],['765-249-9041x18574'],['https://instagram.com/elizabeth09'],"['1568 Johnson Spur Suite 814\nJosephshire, MO...",The relentless pursuit of knowledge often lead...
1997,1997,['heatherbrock@yahoo.com'],"['sarahschaefer', 'jleon']","['JQXY24790887186739', '889-12-2349']",['(603)906-2704'],['https://twitter.com/miguelmoore'],"['061 Victoria Ferry\nNorth Kimmouth, PR 91810']",## The Algorithmic Echo Chamber and the Quest ...
1998,1998,['kelliweber@yahoo.com'],['jacquelinewise'],['758-72-7474'],['394.247.8603x011'],['https://youtube.com/c/kathy09'],"['3185 Moore Parks Apt. 706\nNew Cynthiaburgh,...",The pursuit of knowledge is often a winding ro...


In [3]:
# Read the file

train_df = pd.read_json("/Users/pin.lyu/Documents/BC_Folder/NLP/Data/pii-detection-removal-from-educational-data/train.json")

test_df = pd.read_json("/Users/pin.lyu/Documents/BC_Folder/NLP/Data/pii-detection-removal-from-educational-data/test.json")

In [4]:
# Check the first row's EMAIL value and its type
# List of all PII columns to check
pii_columns = ['EMAIL', 'USERNAME', 'ID_NUM', 'PHONE_NUM', 'URL_PERSONAL', 'STREET_ADDRESS']

# Print the first value and its type for each column
for column in pii_columns:
    first_value = synthetic_df[column].iloc[0]
    print(f"{column}:")
    print(f"  Value: {first_value}")
    print(f"  Type: {type(first_value)}")
    print("-" * 40)

EMAIL:
  Value: ['lisa08@gmail.com']
  Type: <class 'str'>
----------------------------------------
USERNAME:
  Value: ['carrie26']
  Type: <class 'str'>
----------------------------------------
ID_NUM:
  Value: ['711801320', '522 AYO']
  Type: <class 'str'>
----------------------------------------
PHONE_NUM:
  Value: ['+1-875-586-8809x1891']
  Type: <class 'str'>
----------------------------------------
URL_PERSONAL:
  Value: ['https://twitter.com/amanda96']
  Type: <class 'str'>
----------------------------------------
STREET_ADDRESS:
  Value: ['9930 Joy Hollow Suite 517\nSherriport, WI 53987', '22345 Sheri Orchard Suite 279\nLake Hollystad, MT 05112']
  Type: <class 'str'>
----------------------------------------


In [5]:
synthetic_df.head()

,Unnamed: 0,EMAIL,USERNAME,ID_NUM,PHONE_NUM,URL_PERSONAL,STREET_ADDRESS,ESSAY
0,0,['lisa08@gmail.com'],['carrie26'],"['711801320', '522 AYO']",['+1-875-586-8809x1891'],['https://twitter.com/amanda96'],"['9930 Joy Hollow Suite 517\nSherriport, WI 53...",The digital age has blurred the lines between ...
1,1,['tammy76@yahoo.com'],"['nhoward', 'juancampos']",['Z89-24I'],['515-978-1565'],"['https://twitter.com/qgrimes', 'https://twitt...","['097 Sanchez Islands Apt. 393\nPort Tammy, AS...",The digital age has irrevocably woven itself i...
2,2,"['bosborne@gmail.com', 'crystalgarcia@hotmail....","['james71', 'bishoptanner', 'debra94']",['418 3MZ'],"['2679537510', '608-399-3318x868']",['https://instagram.com/ronaldknight'],"['PSC 1611, Box 6207\nAPO AA 90471', '4151 Mic...",The question of identity in the digital age is...
3,3,['yfigueroa@yahoo.com'],['scott92'],['161-16-1975'],['(769)972-8457x6377'],['https://twitter.com/wallacedouglas'],[],The flickering fluorescent lights of the unive...
4,4,"['donnadennis@gmail.com', 'carterhannah@hotmai...",['michelelopez'],['DPLW42035574485291'],['789-542-6223'],['https://facebook.com/karicarter'],"['90611 Robert Plaza\nYangberg, OR 55838']",## The Unexpected Detour: Finding Community in...


In [6]:
def map_tokens_to_entities(row):
    # Convert all entity columns to normalized sets
    target_lists = {
        'EMAIL': {str(x).strip().lower() for x in row['EMAIL'] if pd.notna(x)},
        'USERNAME': {str(x).strip().lower() for x in row['USERNAME'] if pd.notna(x)},
        'ID_NUM': {str(x).strip().lower() for x in row['ID_NUM'] if pd.notna(x)},
        'PHONE_NUM': {str(x).strip().lower() for x in row['PHONE_NUM'] if pd.notna(x)},
        'URL_PERSONAL': {str(x).strip().lower() for x in row['URL_PERSONAL'] if pd.notna(x)},
        'STREET_ADDRESS': {str(x).strip().lower() for x in row['STREET_ADDRESS'] if pd.notna(x)}
    }

    # Tokenization with email support
    tokens = re.findall(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b|'  # Emails
        r'\+?\(?\d[\d\-x\(\)]{5,}\d|'                            # Phones
        r'\bhttps?://[^\s]+|'                                     # URLs
        r'\b[\w\.\-\/\']+',                                       # Words/addresses
        row['ESSAY'],
        flags=re.IGNORECASE
    )

    # Labeling with normalized comparison
    labels = [
        next((etype for etype, eset in target_lists.items() if t.lower().strip() in eset), 'O')
        for t in tokens
    ]

    return pd.Series({'tokens': tokens, 'labels': labels})

# Apply (overwrites existing columns)
synthetic_df[['tokens', 'labels']] = synthetic_df.apply(map_tokens_to_entities, axis=1)

In [7]:
# Get unique labels (since each row contains a list of labels)
unique_labels = set()

# Flatten all labels from all rows
for label_list in synthetic_df['labels']:
    unique_labels.update(label_list)

print("Unique labels found:")
print(unique_labels)

Unique labels found:
{'URL_PERSONAL', 'STREET_ADDRESS', 'ID_NUM', 'EMAIL', 'O', 'USERNAME'}


In [8]:
print("Data types in 'labels':")
print(synthetic_df['labels'].apply(type).value_counts())

print("\nSample unique values:")
print(synthetic_df['labels'].explode().dropna().unique())

Data types in 'labels':
labels
<class 'list'>    2000
Name: count, dtype: int64

Sample unique values:
['O' 'EMAIL' 'ID_NUM' 'USERNAME' 'URL_PERSONAL' 'STREET_ADDRESS']


In [9]:
replacement_map = {
    'EMAIL': 'B-EMAIL',
    'ID_NUM': 'B-ID_NUM',
    'USERNAME': 'B-USERNAME',
    'URL_PERSONAL': 'B-URL_Personal',
    'STREET_ADDRESS': 'B-Address',  # Added based on your unique values
    'O': 'O'  # Keep unchanged
}

# Fast vectorized operation
synthetic_df['labels'] = [
    [replacement_map.get(item, item) for item in sublist] 
    for sublist in synthetic_df['labels']
]

In [10]:
synthetic_df['labels'].head()

0    [O, O, O, O, O, O, O, O, O, O, O, O, B-EMAIL, ...
1    [O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...
2    [O, O, O, O, O, O, O, O, O, B-EMAIL, O, O, O, ...
3    [O, O, O, O, O, O, O, O, O, B-EMAIL, O, O, O, ...
4    [O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...
Name: labels, dtype: object

In [11]:
def detect_trailing_whitespace(text):
    if not isinstance(text, str):
        return []
    
    # Split text into lines
    lines = text.split('\n')
    all_results = []
    
    for line in lines:
        # Skip empty lines
        if not line.strip():
            continue
            
        # Split into words while preserving original spacing
        words = re.split(r'(\s+)', line)
        words = [w for w in words if w]  # Remove empty strings
        
        # Initialize results for this line
        line_results = []
        
        # Track whether we're in whitespace
        for i, word in enumerate(words):
            if word.isspace():
                # Only mark True if this is trailing whitespace
                if i == len(words) - 1:  # Last element in line
                    line_results[-1] = True  # Mark previous word as having trailing space
            else:
                line_results.append(False)
                
        all_results.extend(line_results)
    
    return all_results

# Apply to dataframe
synthetic_df['trailing_whitespace'] = synthetic_df['ESSAY'].apply(detect_trailing_whitespace)

# Convert to comma-separated string
synthetic_df['trailing_whitespace'] = synthetic_df['trailing_whitespace'].apply(
    lambda x: ','.join(['True' if v else 'False' for v in x]) if x else ''
)

# Verify results
print(synthetic_df[['trailing_whitespace']].head())

                                 trailing_whitespace
0  False,False,False,False,False,False,False,Fals...
1  False,False,False,False,False,False,False,Fals...
2  False,False,False,False,False,False,False,Fals...
3  False,False,False,False,False,False,False,Fals...
4  False,False,False,False,False,False,False,Fals...


In [12]:
# Count all 'True' occurrences in the entire column

total_true = synthetic_df['trailing_whitespace'].str.count('True').sum()

print(f"Total 'True' values: {total_true}")

Total 'True' values: 18


In [13]:
# Find rows where trailing_whitespace contains at least one 'True'

true_rows = synthetic_df[synthetic_df['trailing_whitespace'].str.contains('True', na=False)]

# Display the count and sample of such rows

print(f"\nNumber of rows with 'True': {len(true_rows)}")

print("\nSample rows with 'True':")

print(true_rows[['trailing_whitespace']].head())


Number of rows with 'True': 17

Sample rows with 'True':
                                   trailing_whitespace
13   False,False,False,False,False,False,False,Fals...
86   False,False,False,False,False,False,False,Fals...
206  False,False,False,False,False,False,False,Fals...
388  False,False,False,False,False,False,False,Fals...
411  False,False,False,False,False,False,False,Fals...


In [15]:
# output directory 
output_dir = "/Users/pin.lyu/Documents/BC_Folder/NLP/Data/"
synthetic_reformat = synthetic_df[['ESSAY', 'tokens', 'trailing_whitespace', 'labels']]
# CSV
synthetic_reformat.to_csv(os.path.join(output_dir, "synthetic_reformat.csv"), index=False)